# ListT5 Grouping Method Experiment

Small inference-only experiment for the ListT5 paper (`2402.15838v3.pdf`). The paper uses ListT5 with an m-ary tournament sort: candidates are split into sequential groups of `listwise_k=5`, ListT5 selects the winners from each group, and the tournament repeats until the top reranked documents are found.

This notebook keeps the official ListT5 inference and BEIR evaluation code unchanged, and only swaps the grouping phase used inside tournament sort.

Main comparison:
- `sequential`: official contiguous chunking from `run_listt5.py`.
- `score_balanced`: sorts candidates by first-stage rank and distributes them round-robin across groups.
- `random`: seeded random grouping for a sanity-check baseline.

No training is performed. The expected baseline check is Table 2 ListT5-base, BM25 top-100, `r=2`, NDCG@10.

## 1. Setup

Run this notebook from the `Project` directory. It expects the local repo at `Project/ListT5`. A CUDA GPU is strongly recommended because the official evaluator moves the model to `cuda`.

In [ ]:
# Run this cell before the imports below.
# Kaggle usually already includes torch/transformers. Avoid pinning transformers==4.33.3 here
# because its old tokenizers dependency may fail to build on newer Python kernels.
!pip install -q jsonlines sentencepiece huggingface_hub beir

In [ ]:
from pathlib import Path
import math
import random
import sys
import time
from types import SimpleNamespace

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ListT5").exists() and (PROJECT_ROOT.parent / "ListT5").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

LISTT5_ROOT = PROJECT_ROOT / "ListT5"
if not (LISTT5_ROOT / "run_listt5.py").exists() and (LISTT5_ROOT / "ListT5" / "run_listt5.py").exists():
    LISTT5_ROOT = LISTT5_ROOT / "ListT5"
assert (LISTT5_ROOT / "run_listt5.py").exists(), f"Could not find run_listt5.py under {PROJECT_ROOT}"

sys.path.insert(0, str(LISTT5_ROOT))

import torch
from beir_eval import run_rerank_eval
from beir_length_mapping import BEIR_LENGTH_MAPPING
import FiDT5 as fid_module
from run_listt5 import ListT5Evaluator, read_jsonl

# Compatibility patch for newer transformers versions.
# Recent transformers checks tied weights at encoder.embed_tokens, while ListT5 wraps
# the encoder inside EncoderWrapper. Expose the wrapped embedding module there.
if not hasattr(fid_module.EncoderWrapper, "embed_tokens"):
    fid_module.EncoderWrapper.embed_tokens = property(lambda self: self.encoder.embed_tokens)

# Newer transformers passes more positional arguments into T5 blocks than
# the original ListT5 CheckpointWrapper.forward accepted. For inference we
# do not need checkpointing, so forward all args directly to the wrapped block.
def _checkpoint_wrapper_forward_compat(self, *args, **kwargs):
    if self.use_checkpoint and self.training:
        return torch.utils.checkpoint.checkpoint(lambda *inner_args: self.module(*inner_args, **kwargs), *args)
    return self.module(*args, **kwargs)

fid_module.CheckpointWrapper.forward = _checkpoint_wrapper_forward_compat

DATA_DIR = PROJECT_ROOT / "data" / "beir-eval-bm25-top100"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "grouping_method_experiment"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("ListT5 root:", LISTT5_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

## 2. Experiment Knobs

Defaults are chosen to match Table 2 for ListT5-base on BM25 top-100: `listwise_k=5`, `out_k=2`, `topk=100`, `rerank_topk=10`. The dataset list starts with five representative BEIR datasets and can be expanded by adding names from `TABLE2_LISTT5_BASE_TOP100`.

In [ ]:
MODEL_PATH = "Soyoung97/ListT5-base"
HF_DATASET_REPO = "Soyoung97/beir-eval-bm25-top100"

# Set to None for full Table 2-style evaluation. Use a small number, e.g. 10, for a smoke test.
MAX_QUERIES = None

DEFAULT_DATASETS = [
    "trec-covid",
    "nfcorpus",
    "fiqa",
    "scifact",
    "arguana",
]

STRATEGIES = ["sequential", "score_balanced", "random"]
SEEDS = [0]  # Add more seeds for random grouping, e.g. [0, 1, 2].

LISTWISE_K = 5
OUT_K = 2
TOPK = 100
RERANK_TOPK = 10
BATCH_SIZE = 20

# Paper Table 2, BM25 top-100, ListT5-base (r=2), NDCG@10.
TABLE2_LISTT5_BASE_TOP100 = {
    "trec-covid": 0.783,
    "nfcorpus": 0.356,
    "bioasq": 0.564,
    "nq": 0.531,
    "hotpotqa": 0.726,
    "fiqa": 0.396,
    "signal": 0.335,
    "news": 0.485,
    "robust04": 0.521,
    "arguana": 0.489,
    "touche": 0.334,
    "cqadupstack": 0.388,
    "quora": 0.864,
    "dbpedia-entity": 0.437,
    "scidocs": 0.176,
    "fever": 0.798,
    "climate-fever": 0.240,
    "scifact": 0.741,
}

pd.DataFrame(
    [{"dataset": d, "table2_listt5_base_r2_ndcg10": TABLE2_LISTT5_BASE_TOP100[d]} for d in DEFAULT_DATASETS]
)

## 3. Data Helper

The official README uses `Soyoung97/beir-eval-bm25-top100`. This helper reuses a local file if available. The repo already includes `ListT5/trec-covid.jsonl`, so that dataset can run without downloading. Other datasets are downloaded through `huggingface_hub` on first use.

In [ ]:
def write_jsonl(path, rows):
    import jsonlines
    with jsonlines.open(path, "w") as writer:
        writer.write_all(rows)


def dataset_path(dataset_name: str, max_queries: int | None = None) -> Path:
    """Return a local JSONL path for a BEIR top-100 dataset."""
    local_copy = DATA_DIR / f"{dataset_name}.jsonl"
    bundled_copy = LISTT5_ROOT / f"{dataset_name}.jsonl"

    if local_copy.exists():
        base_path = local_copy
    elif bundled_copy.exists():
        base_path = bundled_copy
    else:
        try:
            from huggingface_hub import hf_hub_download
        except ImportError as exc:
            raise ImportError("Install huggingface_hub to download BEIR JSONL files.") from exc

        downloaded = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=f"{dataset_name}.jsonl",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            local_dir_use_symlinks=False,
        )
        base_path = Path(downloaded)

    if max_queries is None:
        return base_path

    subset_path = DATA_DIR / f"{dataset_name}.first{max_queries}.jsonl"
    if not subset_path.exists():
        rows = read_jsonl(str(base_path))[:max_queries]
        write_jsonl(subset_path, rows)
    return subset_path


for dname in DEFAULT_DATASETS:
    print(dname, "->", dataset_path(dname, MAX_QUERIES))

## 4. Grouping Policies

`sequential_groups` is intentionally identical to the official `group2chunks` behavior: contiguous groups of size `listwise_k`. The alternatives below return the same kind of list-of-index-groups, so the rest of `run_listt5.py` stays untouched.

In [ ]:
def sequential_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    return [items[i:i + group_size] for i in range(0, len(items), group_size)]


def score_balanced_groups(items, group_size: int, seed: int = 0):
    """Spread high and low first-stage ranks across groups.

    Candidate index 0 is the first-stage rank 1 document, index 1 is rank 2, etc.
    Sorting by index reconstructs the first-stage order, then round-robin assignment
    prevents one group from containing only the strongest initial candidates.
    """
    items = list(items)
    if len(items) <= group_size:
        return [items]

    ranked = sorted(items)
    n_groups = math.ceil(len(ranked) / group_size)
    groups = [[] for _ in range(n_groups)]
    for i, item in enumerate(ranked):
        groups[i % n_groups].append(item)
    return [group for group in groups if group]


def random_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    rng = random.Random(seed)
    rng.shuffle(items)
    return sequential_groups(items, group_size, seed=seed)


GROUPING_POLICIES = {
    "sequential": sequential_groups,
    "score_balanced": score_balanced_groups,
    "random": random_groups,
}

demo = list(range(20))
pd.concat(
    [
        pd.DataFrame({"strategy": name, "group": i + 1, "indices": [group]})
        for name, fn in GROUPING_POLICIES.items()
        for i, group in enumerate(fn(demo, LISTWISE_K, seed=0))
    ],
    ignore_index=True,
)

## 5. Evaluator Subclass

This is the only integration point. `GroupingListT5Evaluator` inherits the official evaluator and overrides `group2chunks`. Model loading, generation, output scoring, caching, and BEIR metrics are still handled by the original code.

In [ ]:
class GroupingListT5Evaluator(ListT5Evaluator):
    def load_model(self):
        start = time.time()
        print("Loading model..")
        print(f"Loading fid model from {self.args.model_path}")
        model = fid_module.FiDT5.from_pretrained(self.args.model_path, use_safetensors=False).to("cuda")
        model.eval()
        print(f"Done! took {time.time() - start} second")
        return model

    def group2chunks(self, l, n=5):
        strategy = getattr(self.args, "grouping_strategy", "sequential")
        seed = getattr(self.args, "seed", 0)
        try:
            group_fn = GROUPING_POLICIES[strategy]
        except KeyError as exc:
            raise ValueError(f"Unknown grouping strategy: {strategy}") from exc
        yield from group_fn(l, n, seed=seed)


def make_args(dataset_name: str, strategy: str, seed: int = 0, max_queries: int | None = None):
    input_path = dataset_path(dataset_name, max_queries=max_queries)
    subset_tag = "full" if max_queries is None else f"first{max_queries}"
    output_path = OUTPUT_DIR / strategy / f"seed{seed}" / subset_tag / f"{dataset_name}_output.jsonl"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    max_input_length = BEIR_LENGTH_MAPPING.get(dataset_name)
    if max_input_length is None:
        raise ValueError(f"No max input length for dataset {dataset_name}. Add it to BEIR_LENGTH_MAPPING or pass a known BEIR name.")

    return SimpleNamespace(
        firststage_result_key="bm25_results",
        docid_key="docid",
        pid_key="pid",
        qrels_key="qrels",
        score_key="bm25_score",
        question_text_key="q_text",
        text_key="text",
        title_key="title",
        model_path=MODEL_PATH,
        topk=TOPK,
        max_input_length=max_input_length,
        padding="max_length",
        listwise_k=LISTWISE_K,
        rerank_topk=RERANK_TOPK,
        out_k=OUT_K,
        dummy_number=21,
        verbose=False,
        seed=seed,
        bsize=BATCH_SIZE,
        input_path=str(input_path),
        output_path=str(output_path),
        measure_flops=False,
        skip_no_candidate=False,
        skip_issubset=False,
        max_gen_length=LISTWISE_K + 2,
        grouping_strategy=strategy,
    )

## 6. Run One Dataset

Start with one dataset and sequential grouping. If this is a full run, the NDCG@10 should be close to the Table 2 reference for the same dataset. Small differences can happen from dependency or hardware differences; subset runs should not be compared directly to Table 2.

In [ ]:
def output_is_complete(output_path: Path, input_path: str) -> bool:
    if not output_path.exists():
        return False
    try:
        return len(read_jsonl(str(output_path))) == len(read_jsonl(input_path))
    except Exception:
        return False


def run_single(dataset_name: str, strategy: str, seed: int = 0, max_queries: int | None = None, reuse_existing: bool = True):
    args = make_args(dataset_name, strategy, seed=seed, max_queries=max_queries)
    output_path = Path(args.output_path)
    start = time.time()

    if reuse_existing and output_is_complete(output_path, args.input_path):
        ndcg10, metric_text = run_rerank_eval(str(output_path))
        mode = "reused_output"
    else:
        evaluator = GroupingListT5Evaluator(args)
        ndcg10, metric_text = evaluator.run_tournament_sort()
        mode = "new_inference"

    elapsed = time.time() - start
    table2 = TABLE2_LISTT5_BASE_TOP100.get(dataset_name)
    return {
        "dataset": dataset_name,
        "strategy": strategy,
        "seed": seed,
        "max_queries": max_queries,
        "ndcg@10": float(ndcg10),
        "table2_listt5_base_r2": table2,
        "delta_vs_table2": None if table2 is None or max_queries is not None else float(ndcg10) - table2,
        "seconds": elapsed,
        "mode": mode,
        "output_path": str(output_path),
    }


# Smoke check: uncomment to run one small subset quickly.
# run_single("trec-covid", "sequential", max_queries=5, reuse_existing=True)

## 7. Run the Main Comparison

This cell runs each grouping method for the configured datasets. For Table 2 matching, keep `MAX_QUERIES = None` and include `sequential` in `STRATEGIES`.

In [ ]:
def run_grid(datasets=DEFAULT_DATASETS, strategies=STRATEGIES, seeds=SEEDS, max_queries=MAX_QUERIES):
    rows = []
    for dataset_name in datasets:
        for strategy in strategies:
            strategy_seeds = seeds if strategy == "random" else [seeds[0]]
            for seed in strategy_seeds:
                print(f"\n=== {dataset_name} | {strategy} | seed={seed} ===")
                rows.append(run_single(dataset_name, strategy, seed=seed, max_queries=max_queries))
                display(pd.DataFrame(rows).tail(1))
    return pd.DataFrame(rows)


# This is the main experiment call.
results = run_grid()
results

## 8. Summary Tables

The first table checks whether official sequential grouping reproduces the paper baseline. The second table compares grouping methods directly on NDCG@10.

In [ ]:
baseline_check = (
    results[results["strategy"] == "sequential"]
    [["dataset", "ndcg@10", "table2_listt5_base_r2", "delta_vs_table2", "output_path"]]
    .sort_values("dataset")
)
baseline_check

In [ ]:
comparison = results.pivot_table(
    index="dataset",
    columns="strategy",
    values="ndcg@10",
    aggfunc="mean",
)

if "sequential" in comparison.columns:
    for col in comparison.columns:
        comparison[f"{col}_minus_sequential"] = comparison[col] - comparison["sequential"]

comparison.reset_index()

## 9. Optional: Expand to More Datasets

To expand beyond the first five datasets, set `DEFAULT_DATASETS` to any subset of the keys below and rerun Sections 7 and 8. Keep `MAX_QUERIES = None` for full comparable numbers.

In [ ]:
pd.DataFrame(
    [{"dataset": name, "table2_listt5_base_r2_ndcg10": score, "max_input_length": BEIR_LENGTH_MAPPING.get(name)}
     for name, score in TABLE2_LISTT5_BASE_TOP100.items()]
).sort_values("dataset")

## 10. Notes for Reporting

- The independent variable is only `grouping_strategy`; all other ListT5 inference parameters stay fixed.
- Use the sequential full-run row as the reproducibility check against Table 2.
- Report `score_balanced - sequential` per dataset to make the grouping effect visible.
- Treat `random` as a sanity-check baseline. If using random grouping, run multiple seeds and report mean and standard deviation.
- Subset runs are useful for debugging but should not be used as Table 2 replication evidence.